In [ ]:
!pip install gradio tensorflow scikit-learn opencv-python scikit-image pillow pandas matplotlib seaborn -q

import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import gradio as gr
from PIL import Image
import shutil
print("Setup complete!")

Setup complete!


In [ ]:
from google.colab import files

# Create directories
os.makedirs('dataset/train', exist_ok=True)
os.makedirs('dataset/test', exist_ok=True)
os.makedirs('uploads', exist_ok=True)

print("Upload your dataset (zipped or individual images). Recommended: Kaggle Minerals or core photos.")
uploaded = files.upload()

# Example: If zipped
import zipfile
for fn in uploaded.keys():
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn, 'r') as zip_ref:
            zip_ref.extractall('dataset')
print("Dataset uploaded! Organize into subfolders like dataset/train/quartz/, dataset/train/pyrite/ etc.")

Upload your dataset (zipped or individual images). Recommended: Kaggle Minerals or core photos.


Saving parthara.zip to parthara.zip
Dataset uploaded! Organize into subfolders like dataset/train/quartz/, dataset/train/pyrite/ etc.


In [ ]:
def preprocess_image(img_path, target_size=(224, 224)):
    img = cv2.imread(img_path)
    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Basic enhancements
    img = cv2.GaussianBlur(img, (5,5), 0)
    return img / 255.0

def extract_features(img):
    # Color features
    hsv = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2HSV)
    color_hist = cv2.calcHist([hsv], [0], None, [8], [0, 256]).flatten()

    # Texture (Haralick or simple GLCM via skimage)
    from skimage.feature import graycomatrix, graycoprops
    gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    glcm = graycomatrix(gray, [1], [0], 256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0,0]

    return np.concatenate([color_hist, [contrast]])

print("Feature functions ready.")

Feature functions ready.


In [ ]:
# Load sample images for clustering demo

image_dir = 'dataset'
image_paths = []

# Recursively find all image files in the dataset directory
for root, _, files in os.walk(image_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
            image_paths.append(os.path.join(root, file))


if not image_paths:
    print(f"Warning: No image files found in '{image_dir}'. Please ensure your dataset is correctly organized.")

    if os.path.exists('DCID-7.jpg'):
        image_paths = ['DCID-7.jpg']
        print("Using 'DCID-7.jpg' as a fallback since no dataset images were found.")
    else:
        raise FileNotFoundError(f"No image files found in '{image_dir}' and 'DCID-7.jpg' is also not present.")


features = []

for p in image_paths[:100]:
    img = preprocess_image(p)
    feat = extract_features(img)
    features.append(feat)

features = np.array(features)

# Dynamically set n_clusters to be at most the number of samples, and at least 1
num_samples = len(features)
if num_samples == 0:
    print("No features extracted for clustering. KMeans will not be run.")
    kmeans = None
else:
    n_clusters_to_use = min(num_samples, 5) # Use at most 5 clusters, or fewer if not enough samples
    kmeans = KMeans(n_clusters=n_clusters_to_use, random_state=42, n_init='auto').fit(features)  # e.g., 5 mineral groups
    print(f"Unsupervised clusters ready ({n_clusters_to_use} clusters). Use for discovering similar samples.")

Unsupervised clusters ready (5 clusters). Use for discovering similar samples.


In [ ]:
# ====================== CLASS NAMES & MODEL DEFINITION ======================
# All 7 minerals the model can classify
class_names = [
    'biotite',
    'bornite',
    'chrysocolla',
    'malachite',
    'muscovite',
    'pyrite',
    'quartz'
]

num_classes = len(class_names)  # 7
print(f"num_classes set to {num_classes}: {class_names}")

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
output = Dense(num_classes, activation='softmax')(x)
model = Model(base_model.input, output)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print("Model compiled. Output layer:", model.output_shape)


num_classes set to 7: ['biotite', 'bornite', 'chrysocolla', 'malachite', 'muscovite', 'pyrite', 'quartz']
Model compiled. Output layer: (None, 7)


In [ ]:
# ====================== MINERAL DATABASE ======================
mineral_db = {
    'biotite':     {'color': 'Black/dark brown', 'hardness': '2.5-3',   'texture': 'Platy, micaceous cleavage', 'common_in': 'Granites, schists, gneisses'},
    'bornite':     {'color': 'Copper-red to purple iridescent', 'hardness': '3',     'texture': 'Massive, tarnished iridescent', 'common_in': 'Copper ore deposits, porphyry Cu'},
    'chrysocolla': {'color': 'Blue-green', 'hardness': '2-4',   'texture': 'Earthy, amorphous', 'common_in': 'Oxidized copper zones'},
    'malachite':   {'color': 'Bright green', 'hardness': '3.5-4', 'texture': 'Banded, botryoidal', 'common_in': 'Oxidized copper ore zones'},
    'muscovite':   {'color': 'Silver/pale yellow', 'hardness': '2-2.5', 'texture': 'Pearly sheen, thin flexible sheets', 'common_in': 'Granites, schists, pegmatites'},
    'pyrite':      {'color': 'Brassy yellow', 'hardness': '6-6.5', 'texture': 'Cubic crystals, striated faces', 'common_in': 'Sulfide ore deposits'},
    'quartz':      {'color': 'White/gray/colorless', 'hardness': '7',     'texture': 'Glassy, conchoidal fracture', 'common_in': 'Veins, host rock, sandstone'},
}

# ====================== MINERAL IDENTIFICATION FUNCTION ======================
def identify_mineral(image):
    """
    Identifies a mineral from an image.
    - If model weights are trained: uses CNN prediction (uncomment training block below)
    - If no trained weights yet: uses color-histogram heuristic as a reasonable baseline
    Returns: (mineral_name, properties_dict, visual_feature_array)
    """
    # Preprocess
    if isinstance(image, str):
        img_proc = preprocess_image(image)
    else:
        # Gradio passes numpy array already (uint8 or float)
        img_proc = cv2.resize(image, (224, 224))
        if img_proc.dtype != np.float32 and img_proc.max() > 1:
            img_proc = img_proc / 255.0



    # ---- Color-heuristic fallback (works without training) ----
    hsv = cv2.cvtColor((img_proc * 255).astype(np.uint8), cv2.COLOR_RGB2HSV)
    mean_h = float(np.mean(hsv[:,:,0]))   # Hue  0-179
    mean_s = float(np.mean(hsv[:,:,1]))   # Saturation 0-255
    mean_v = float(np.mean(hsv[:,:,2]))   # Value (brightness) 0-255

    # Simple heuristic rules based on dominant color signature
    if mean_s < 30 and mean_v > 160:
        mineral = 'quartz'         # Low saturation, bright white/gray
    elif mean_s < 30 and mean_v < 100:
        mineral = 'biotite'        # Low saturation, dark
    elif 10 <= mean_h <= 25 and mean_s > 80:
        mineral = 'bornite'        # Copper-red / orange hue range
    elif 35 <= mean_h <= 90 and mean_s > 60:
        if mean_s > 130:
            mineral = 'malachite'   # Bright saturated green
        else:
            mineral = 'chrysocolla' # Blue-green, lower saturation
    elif 85 <= mean_h <= 130 and mean_s > 40:
        mineral = 'chrysocolla'    # Blue-green
    elif 20 <= mean_h <= 35 and mean_s > 60 and mean_v > 120:
        mineral = 'pyrite'         # Brassy/yellow
    elif mean_s < 50 and 100 < mean_v < 200:
        mineral = 'muscovite'      # Silvery/pale, low saturation mid-bright
    else:
        mineral = 'quartz'         # Default fallback

    confidence = None  # Will be real number once CNN is trained

    feats = mineral_db.get(mineral, {})
    vis_feats = extract_features(img_proc)
    return mineral, feats, vis_feats, mean_h, mean_s, mean_v

print("Mineral DB and identify_mineral() ready.")
print(f"Supported minerals: {list(mineral_db.keys())}")


Mineral DB and identify_mineral() ready.
Supported minerals: ['biotite', 'bornite', 'chrysocolla', 'malachite', 'muscovite', 'pyrite', 'quartz']


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
import io
from PIL import Image

# ====================== PIXEL HIGHLIGHTING FUNCTION ======================
def generate_mineral_heatmap(original_image, model, class_idx):
    """
    Generates heatmap showing which pixels the model focuses on for the predicted mineral.
    Works with CNN models.
    """
    # Ensure image is (1, 224, 224, 3)
    if len(original_image.shape) == 3:
        img_array = np.expand_dims(original_image, axis=0)
    else:
        img_array = original_image

    # Get the last convolutional layer
    last_conv_layer = None
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv_layer = layer.name
            break

    if last_conv_layer is None:
        # Fallback: Simple color-based mask if no CNN
        gray = cv2.cvtColor((original_image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
        heatmap = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
        return heatmap

    # Create Grad-CAM model
    grad_model = Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_layer).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(pooled_grads, conv_outputs), axis=-1)
    heatmap = np.maximum(heatmap, 0) / np.max(heatmap) if np.max(heatmap) != 0 else heatmap
    heatmap = cv2.resize(heatmap.numpy(), (original_image.shape[1], original_image.shape[0]))

    # Apply colormap
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # Overlay on original
    overlay = cv2.addWeighted(
        (original_image * 255).astype(np.uint8), 0.6,
        heatmap, 0.4, 0
    )

    return overlay


# ====================== COMPARISON FUNCTION ======================
def create_comparison_visual(original_image, predicted_mineral, model, class_names):
    """
    Returns side-by-side comparison: Original | Highlighted Regions
    """
    try:
        class_idx = class_names.index(predicted_mineral)

        # Generate highlighted image
        highlighted = generate_mineral_heatmap(original_image, model, class_idx)

        # Create figure
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        axes[0].imshow(original_image)
        axes[0].set_title("Original Image")
        axes[0].axis('off')

        axes[1].imshow(highlighted)
        axes[1].set_title(f"Model Focus - {predicted_mineral} Regions")
        axes[1].axis('off')

        plt.suptitle(f"Pixel-wise Mineral Identification: {predicted_mineral}", fontsize=14)
        plt.tight_layout()

        # Convert to PIL Image for Gradio
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        buf.seek(0)
        comparison_img = Image.open(buf)

        plt.close(fig)
        return comparison_img

    except Exception as e:
        # Fallback simple visualization
        fallback = original_image.copy()
        cv2.putText(fallback, f"Predicted: {predicted_mineral}",
                   (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        return Image.fromarray(fallback.astype(np.uint8))

In [ ]:
# ====================== GRADIO INTERFACE ======================
def process_upload(image):
    """
    Main prediction pipeline called by Gradio.
    Returns: predicted mineral, mineral properties, core logging summary
    """
    if image is None:
        return "No image provided", "", ""

    mineral, props, vis_feats, h, s, v = identify_mineral(image)

    # Format properties nicely
    props_str = "\n".join([f"  {k.capitalize()}: {val}" for k, val in props.items()])

    # Core logging summary
    logging_str = (
        f"Identified Mineral : {mineral.upper()}\n"
        f"Color              : {props.get('color', 'N/A')}\n"
        f"Hardness (Mohs)    : {props.get('hardness', 'N/A')}\n"
        f"Texture            : {props.get('texture', 'N/A')}\n"
        f"Common In          : {props.get('common_in', 'N/A')}\n"
        f"---\n"
        f"HSV Analysis       : H={h:.1f}  S={s:.1f}  V={v:.1f}\n"
        f"Color Contrast     : {vis_feats[-1]:.3f}\n"
        f"Note: Using color-heuristic baseline. Train ResNet50 on labelled dataset for CNN predictions."
    )

    return mineral.upper(), props_str, logging_str

# ---- Gradio UI ----
iface = gr.Interface(
    fn=process_upload,
    inputs=gr.Image(type="numpy", label="Upload Rock / Core Sample Image"),
    outputs=[
        gr.Textbox(label="Predicted Mineral"),
        gr.Textbox(label="Mineral Properties", lines=6),
        gr.Textbox(label="Core Logging Summary", lines=10)
    ],
    title="Automatic Core Sample Logging & Mineral Identifier",
    description=(
        "Upload a drill core or rock image for ML-based mineral identification.\n"
        "Supported minerals: Biotite | Bornite | Chrysocolla | Malachite | Muscovite | Pyrite | Quartz"
    ),
    examples=[],
    flagging_mode="never"
)

iface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dd55de55ddf9ca507f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
